> 在线讲义：[第 20 章 · 前馈网络 FFN：每个词回到自己座位上消化](https://wikieden.github.io/transformer-visual/lessons/20-ffn.html) · 本书《从零读懂 Transformer》

# 第 20 章 · 前馈网络 FFN：每个词回到自己座位上消化

配套讲义：[在线版](https://wikieden.github.io/transformer-visual/lessons/20-ffn.html) · 仓库里 `transformer-deep-dive/notes/lessons/20-ffn.md`

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 torch 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)  # 打印保留 3 位小数，不用科学计数法
print("torch", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 / 说法 | PyTorch 写法 | 跑出来 |
|---|---|---|---|
| ReLU | $\max(0, z)$ | `torch.relu(z)` | [3, −1, 0.5, −2] → [3, 0, 0.5, 0] |
| FFN | $\max(0, xW_1+b_1)W_2+b_2$ | `nn.Sequential(Linear, ReLU, Linear)` | 3 × 512 → 3 × 2048 → 3 × 512 |
| 逐个位置 | 每个词单独算 | 整句 vs 一行一行 | 完全一样 |
| 参数 | $W_1$、$W_2$ 各 512 × 2048 | `sum(p.numel() ...)` | 不算 bias 2,097,152 ≈ 210 万 |
| 两次转换 = 一次 | $xW_1W_2 = x(W_1W_2)$ | `(v @ w1) @ w2` vs `v @ (w1 @ w2)` | 相等；夹 ReLU 后不相等 |
| sigmoid、SiLU | $\sigma(x)$、$x\sigma(x)$ | `torch.sigmoid`、`F.silu` | x = −1：0.27、−0.27 |
| GLU 闸门 | $\odot$ | `gate * content` | [9.8, 5, 0.2] |
| SwiGLU 的 $d_{ff}$ | $\tfrac83 d_{model}$ 取 64 的倍数 | `swiglu_d_ff(512)` | 1344；参数 2,064,384，和原版差不多 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn

**函数 `ffn`**：原论文：max(0, x W1 + b1) W2 + b2。

In [ ]:
def ffn(d_model: int = 512, d_ff: int = 2048, bias: bool = True) -> nn.Sequential:
    """原论文：max(0, x W1 + b1) W2 + b2。"""
    return nn.Sequential(nn.Linear(d_model, d_ff, bias=bias), nn.ReLU(), nn.Linear(d_ff, d_model, bias=bias))

**类 `SwiGLU`**：W2( SiLU(W1 x) ⊙ W3 x )，三个矩阵、没有 bias。

In [ ]:
class SwiGLU(nn.Module):
    """W2( SiLU(W1 x) ⊙ W3 x )，三个矩阵、没有 bias。"""

    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

**函数 `swiglu_d_ff`**：≈ 8/3 · d_model，取最近的 64 的倍数。

In [ ]:
def swiglu_d_ff(d_model: int, multiple: int = 64) -> int:
    """≈ 8/3 · d_model，取最近的 64 的倍数。"""
    return round(8 * d_model / 3 / multiple) * multiple

**函数 `n_params`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def n_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters())

**函数 `r`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def r(t: torch.Tensor, n: int = 2) -> list[float]:
    return [round(float(v), n) + 0.0 for v in t.flatten()]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
z = torch.tensor([3.0, -1.0, 0.5, -2.0])
print("ReLU", z.tolist(), "→", torch.relu(z).tolist())
f = ffn()
torch.manual_seed(0)
x = torch.randn(3, 512)  # "狗咬人" 3 个词
with torch.no_grad():
    h = f[0](x)
    print("\n形状: 进", tuple(x.shape), "→ W1 后", tuple(h.shape), "→ 出", tuple(f(x).shape))
    print(
        "逐个位置：整句一起算 == 每个词单独算:", torch.allclose(f(x), torch.stack([f(row) for row in x]), atol=1e-6)
    )
print("W1、W2 各", f"{512 * 2048:,}", "个数；不算 bias 一共", f"{n_params(ffn(bias=False)):,}")

In [ ]:
print("\n为什么要夹 ReLU")
w1, w2 = torch.randn(4, 8), torch.randn(8, 4)
v = torch.randn(4)
print("  两次转换 == 一次转换 (x W1) W2 == x (W1 W2):", torch.allclose((v @ w1) @ w2, v @ (w1 @ w2), atol=1e-5))
print("  夹 ReLU 以后还相等吗:", torch.allclose(torch.relu(v @ w1) @ w2, v @ (w1 @ w2), atol=1e-5))
for t in [1.0, -1.0]:
    print(f"  3 × max(0, 2x)，x = {t:>4}: {3 * max(0.0, 2 * t)}")
s = torch.tensor([4.0, 1.0, 0.0, -1.0, -4.0])

In [ ]:
print("\nx      :", s.tolist())
print("sigmoid:", r(torch.sigmoid(s)))
print("SiLU   :", r(s * torch.sigmoid(s)), " F.silu 一样:", torch.allclose(F.silu(s), s * torch.sigmoid(s)))
print("ReLU   :", r(torch.relu(s)))
gate, content = torch.tensor([0.98, 0.5, 0.02]), torch.tensor([10.0, 10, 10])

In [ ]:
print("\nGLU 闸门 ⊙ 内容:", r(gate * content, 1), "  点积是", round(float(gate @ content), 1), "（一个数）")
d_ff = swiglu_d_ff(512)
print(f"\nSwiGLU：8/3 × 512 ≈ {8 * 512 / 3:.1f}，取 64 的倍数 → {d_ff}")
print(f"  参数：原版 {n_params(ffn(bias=False)):,}，SwiGLU {n_params(SwiGLU(512, d_ff)):,}（差不多）")
print("  SwiGLU 输出形状:", tuple(SwiGLU(512, d_ff)(x).shape))

## 3. 代码里才看得到的

1. **"逐个位置"在代码里是白送的**：`nn.Linear` 只对最后一维（512 个数）做乘法，前面有几个词都一样处理，所以整句一起算和一个词一个词算，结果完全相同。
2. **`nn.Sequential`**：把几层按顺序串起来，进去的数依次穿过每一层。原版 FFN 正好是"线性 → ReLU → 线性"三步。
3. **SwiGLU 要自己写一个 `nn.Module`**：它有两条路（闸门、内容）再逐个相乘，不是简单串起来，`Sequential` 装不下。CS336 a1 要写的就是这个类。

## 4. 自己改一改

- 把上面任意一格里的数字改一改，再按 Shift + Enter，看结果怎么变。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 开会靠 attention，消化靠 FFN；每个词各算各的，全班同一套方法；
> 先摊开四倍，过门卫负数变 0，再收回原长好相加；
> 两次转换连着做等于一次，中间夹个弯才有用；
> 新版换 SiLU 加闸门，三个矩阵各变窄，八分之三保预算。